In [0]:
%pip install lightgbm catboost optuna

In [0]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px # one-liner charts, high-level
import plotly.graph_objects as go # full control chart

from sklearn.model_selection import StratifiedKFold, cross_validate, cross_val_score, cross_val_score
from sklearn.metrics import roc_auc_score, classification_report, confusion_matrix

from sklearn.preprocessing import StandardScaler, OneHotEncoder, TargetEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier, Pool

import optuna

In [0]:
TARGET = "Exited"

categorical_columns = ["Geography", "Gender"]
numerial_columns = [
    "CreditScore",
    "Age",
    "Tenure",
    "Balance",
    "NumOfProducts",
    "HasCrCard",
    "IsActiveMember",
    "EstimatedSalary",
]

drop_columns  = ["id", "CustomerId"]
submission_columns = ["id", "Exited"]

In [0]:
def load_data():
    train_df = pd.read_csv('train.csv')
    test_df = pd.read_csv('test.csv')

    train_df['source'] = 'train'
    test_df['source'] = 'test'

    df = pd.concat([train_df, test_df], ignore_index=True)
    
    return train_df, test_df, df

In [0]:
def data_spliter(df):
    train = df[df['source'] == 'train'].drop(columns=drop_columns)
    submission_df = df[df['source'] == 'test'][submission_columns]

    X_train = train_df.drop(columns=[TARGET])
    y_train = train_df[TARGET]

    return X_train, y_train, submission_df

In [0]:
train_df, test_df, df = load_data()

# EDA

In [0]:
len(df)

-   **Customer** ID: Уникальный идентификатор каждого клиента.
-   **Surname**: Фамилия клиента.
-   **Credit Score**: Числовое значение, представляющее кредитный рейтинг клиента.
-   **Geography**: Страна проживания клиента (Франция, Испания или Германия).
-   **Gender**: Пол клиента (Мужской или Женский).
-   **Age**: Возраст клиента.
-   **Tenure**: Количество лет, которое клиент обслуживается в банке.
-   **Balance**: Баланс на счёте клиента.
-   **NumOfProducts**: Количество банковских продуктов, которыми пользуется клиент (например, сберегательный счёт, кредитная карта).
-   **HasCrCard**: Наличие кредитной карты у клиента (1 = да, 0 = нет).
-   **IsActiveMember**: Является ли клиент активным членом банка (1 = да, 0 = нет).
-   **EstimatedSalary**: Предполагаемая заработная плата клиента.
-   **Exited**: Ушёл ли клиент (1 = да, 0 = нет).


In [0]:
df.info()

In [0]:
df.describe().T

In [0]:
df.isna().sum()

In [0]:
df.head()

## Pairplot

In [0]:
fig = sns.pairplot(
    data=train_df[numerial_columns + [TARGET]],
    hue=TARGET,
    diag_kind="kde",
    plot_kws={"alpha": 0.5, "s": 15, "edgecolor": None},
    diag_kws={"fill": True, "alpha": 0.6},
    palette={0: "#2196F3", 1: "#FF5722"},
    # corner=True,
)
fig.figure.suptitle("Pairplot of Numerical Features by Churn Status", y=1.02, fontsize=16, fontweight="bold")

## Surnames

In [0]:
display(df['Surname'].value_counts().reset_index())

In [0]:
df[df['Surname'].str.endswith('ov') | df['Surname'].str.endswith('ova') | df['Surname'].str.endswith('ev') | df['Surname'].str.endswith('eva')]

In [0]:
df[df['Surname'].str.contains("?", regex=False)]

In [0]:
display(df[df['Surname'].str.contains("'", regex=False)])

## Balance

In [0]:
plt.figure(figsize=(16,9))
sns.histplot(
    data=train_df,
    x="Balance",
    hue='Exited',
    bins=50,
    multiple='fill',
)

## Salary

In [0]:
plt.figure(figsize=(10,5))
sns.histplot(
    data=train_df,
    x="EstimatedSalary",
    hue='Exited',
    bins=50,
    multiple='fill',
)

## Age

In [0]:
plt.figure(figsize=(12,5))
ax = sns.countplot(
    data=train_df,
    x=df['Age'].astype(int),
    hue='Exited',
)

In [0]:
plt.figure(figsize=(10,8))
sns.histplot(
    data=train_df,
    x="Age",
    hue='Exited',
    bins=60,
    multiple='fill',
)
plt.xticks(np.arange(18,75))
plt.tight_layout()